<a href="https://colab.research.google.com/github/vudinhhoan213/Lab-LSTM/blob/main/Pytorch_LSTM_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Author: ProtonX Team

Website: https://protonx.io/

![](https://storage.googleapis.com/mle-courses-prod/users/61b6fa1ba83a7e37c8309756/private-files/fc210df0-547b-11ef-bf69-71eafa46c86b-RAGGraph___Neo4J___CamelAI__4_.png)

In [2]:
!pip install torchtext

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 34.1 MB/s eta 0:00:00


In [3]:
!pip uninstall torch torchtext -y

Found existing installation: torch 2.8.0+cu126
Uninstalling torch-2.8.0+cu126:
  Successfully uninstalled torch-2.8.0+cu126
Found existing installation: torchtext 0.18.0
Uninstalling torchtext-0.18.0:
  Successfully uninstalled torchtext-0.18.0


In [4]:
pip install torch==2.1.0 torchtext==0.16.0

ERROR: Could not find a version that satisfies the requirement torch==2.1.0 (from versions: 2.2.0, 2.2.1, 2.2.2, 2.3.0, 2.3.1, 2.4.0, 2.4.1, 2.5.0, 2.5.1, 2.6.0, 2.7.0, 2.7.1, 2.8.0, 2.9.0)
ERROR: No matching distribution found for torch==2.1.0


In [5]:
import json
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchtext.vocab import build_vocab_from_iterator
from torchtext.data.utils import get_tokenizer
from sklearn.model_selection import train_test_split
from torchtext.transforms import VocabTransform, ToTensor, PadTransform

# Load data
!wget --no-check-certificate https://storage.googleapis.com/learning-datasets/sarcasm.json -O /tmp/sarcasm.json

with open("/tmp/sarcasm.json", 'r') as f:
    datastore = json.load(f)

# Prepare data
dataset = [item["headline"] for item in datastore]
label_dataset = [item["is_sarcastic"] for item in datastore]

# Tokenization and vocabulary building
tokenizer = get_tokenizer("basic_english")
def yield_tokens(data_iter):
    for text in data_iter:
        yield tokenizer(text)

ModuleNotFoundError: No module named 'torchtext'

In [ ]:
dataset[1]

In [ ]:
label_dataset[1]

In [ ]:
vocab = build_vocab_from_iterator(yield_tokens(dataset), specials=["<unk>", "<pad>", "<bos>", "<eos>"])
vocab.set_default_index(vocab["<unk>"])

In [ ]:
len(vocab)

In [ ]:
vocab["<pad>"]

In [ ]:

max_length = 25
embedding_size = 64

# Define transformations
text_transform = VocabTransform(vocab)
tensor_transform = ToTensor(padding_value=vocab["<pad>"])
pad_transform = PadTransform(max_length, pad_value=vocab["<pad>"])

In [ ]:
def process_text(text):
    return pad_transform(tensor_transform(text_transform(tokenizer(text))))

In [ ]:



# Prepare datasets
class SarcasmDataset(Dataset):
    def __init__(self, sequences, labels):
        # Tất cả các mẫu
        self.sequences = [process_text(seq) for seq in sequences]
        self.labels = torch.tensor(labels, dtype=torch.float32)

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        return self.sequences[idx], self.labels[idx]

In [ ]:
# Custom collate function
def collate_fn(batch):
    sequences, labels = zip(*batch)
    sequences_padded = nn.utils.rnn.pad_sequence(sequences, batch_first=True, padding_value=vocab["<pad>"])
    labels = torch.tensor(labels, dtype=torch.float32)
    return sequences_padded, labels

In [ ]:
# Split data
train_sequences, test_sequences, train_labels, test_labels = train_test_split(
    dataset, label_dataset, test_size=0.2, random_state=42)


In [ ]:
train_sequences[0], train_labels[0]

In [ ]:
train_sequences[1], train_labels[1]

In [ ]:
train_dataset = SarcasmDataset(train_sequences, train_labels)
test_dataset = SarcasmDataset(test_sequences, test_labels)

In [ ]:
train_dataset[0]

In [ ]:
train_dataset[1]

In [ ]:
train_dataset[2]

In [ ]:
len(train_dataset[0][0])

In [ ]:
len(train_dataset)

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True)
test_loader = DataLoader(
    test_dataset, batch_size=32, shuffle=False)

In [ ]:
# Define the custom LSTM model
class CustomLSTMCell(nn.Module):
    def __init__(self, input_size, hidden_size):
        super(CustomLSTMCell, self).__init__()
        self.input_size = input_size # 64
        self.hidden_size = hidden_size # 128

        self.forget_gate = nn.Linear(input_size + hidden_size, hidden_size)
        self.input_gate = nn.Linear(input_size + hidden_size, hidden_size)
        self.output_gate = nn.Linear(input_size + hidden_size, hidden_size)
        self.cell_gate = nn.Linear(input_size + hidden_size, hidden_size)

    def forward(self, x, hidden):

        # h_0, c_0
        h_prev, c_prev = hidden

        combined = torch.cat((x, h_prev), 1)
        f_t = torch.sigmoid(self.forget_gate(combined))
        i_t = torch.sigmoid(self.input_gate(combined))
        o_t = torch.sigmoid(self.output_gate(combined))
        c_tilde = torch.tanh(self.cell_gate(combined))

        # print(f_t, i_t, o_t)

        c_t = f_t * c_prev + i_t * c_tilde
        h_t = o_t * torch.tanh(c_t)

        # h_1, c_1
        return h_t, c_t


class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim):
        super(LSTMClassifier, self).__init__()
        self.embedding = nn.Embedding(
            vocab_size, embedding_dim
        ) # (vocab_size, embedding_size)
        self.hidden_dim = hidden_dim # 128
        self.lstm_cell = CustomLSTMCell(embedding_dim, hidden_dim)
        self.fc = nn.Linear(hidden_dim, output_dim)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # (32, 25)
        # batch_size = 32, seq_len=25
        batch_size, seq_len = x.size()
        embedded = self.embedding(x) # x (32, 25) -> (32, 25, 64)

        h_t = torch.zeros(batch_size, self.hidden_dim).to(x.device) # (32, 128)
        c_t = torch.zeros(batch_size, self.hidden_dim).to(x.device) # (32, 128)


        for t in range(seq_len): # 25
            # embedded[:, t, :] -> (32, 64)
            h_t, c_t = self.lstm_cell(embedded[:, t, :], (h_t, c_t))



        out = self.fc(h_t)
        return self.sigmoid(out)



In [ ]:
# Hyperparameters
hidden_dim = 128
output_dim = 1
device = torch.device(
    'cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
device

In [ ]:
model = LSTMClassifier(
    len(vocab),
    embedding_size,
    hidden_dim,
    output_dim).to(device)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.0005)

In [ ]:
import numpy as np


In [ ]:
# Training the model
epochs = 10
for epoch in range(epochs):
    model.train()
    for sequences, labels in train_loader:
        # (32, 25), (32, )
        sequences, labels = sequences.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(sequences)
        loss = criterion(outputs.squeeze(), labels)
        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch+1}/{epochs}, Loss: {loss.item()}")

    # Validation
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for sequences, labels in test_loader:
            sequences, labels = sequences.to(device), labels.to(device)
            outputs = model(sequences)
            predicted = (outputs.squeeze() > 0.5).float()
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = correct / total
    print(f"Accuracy: {accuracy:.4f}")